# MiniMax H3 (DaSiWa cMMH3 V19) — MANUEL (ComfyUI arayüzünde elle) — Colab

ComfyUI'yi tünelle açar, grafiği **sen elle sürersin**. Bu bir *deneme*: MiniMax H3'ün queen-editor'e alınmaya değip değmediği burada görülecek (v5 yol haritası, madde 213).

**Input:** bir fotoğraf + prompt (ComfyUI UI'da) · **Output:** sesli video — H3 videoyu ve sesi birlikte üretir

**Gerekenler:** **A100** runtime · Colab **Secrets**'ta `CIVITAI_COOKIE` (notebook erişimi açık).

Sıra:
1. **CONFIG** — Civitai cookie'si Colab Secrets'tan okunur
2. **Ortak Yardımcılar** — log + fail-loud run + model doğrulama
3. **ComfyUI + Manager + custom node'lar** (6)
4. **Modeller** — ~40,1 GB; önce gated probe, sonra indirme
5. **Başlat + cloudflared tünel** → UI linki

> Drive kullanılmaz; modeller her oturumda kaynaktan iner (Colab geçici diski).
>
> **Runtime → Change runtime type → A100.** 21 GB'lık model + 15 GB'lık metin kodlayıcı.
>
> **Run all** → en alttaki linke gir → bu klasördeki `workflow.json`'u sürükle-bırak → grafik **I2VA** modunda açılır → Director'e fotoğraf + prompt → **Queue Prompt** (Ctrl+Enter).
>
> Hangi dosya neden iniyor: **`indirilecekler.md`** · modlar, ayarlar, tuzaklar: **`instructions.md`**

## 1) CONFIG

Doldurulacak bir şey yok → **Run all**.

Civitai cookie'si **Colab Secrets**'tan okunur (`CIVITAI_COOKIE`, queen-editor'ün kullandığı sır) — defterin içinde durmaz. Süresi ~30 günde dolar; bittiğinde `civitai.red`'den yeni değeri alıp **sırrı** güncelle.

**Drive mount yok.** `video_experiments/` arayüz denemeleri Drive kullanmıyor; modeller Colab'ın geçici diskine iner. Standardın "önce Drive mount" kuralı Drive kullanan defterler için — burada beklemesi gereken bir auth istemi yok.

In [ ]:
# === CONFIG ===
# Civitai login-gated download: read from Colab Secrets under the same name Queen Editor uses, so
# one paste serves every notebook and the token is not committed with this one.
# How to get the value: civitai.red -> log in -> F12 -> Application -> Cookies -> __Secure-civ-token
# (double-click -> Ctrl+A -> Ctrl+C; a single click truncates it and the len > 200 gate still passes).
# NOTE: auth moved to auth.civitai.com -> the cookie NAME is __Secure-civ-token (NOT the old
#   __Secure-civitai-token) and the value is a short ES256 JWT (~420 chars), not the old long JWE.
# Cookie only; never a ?token= API key -> gated assets answer 401. exp ~30 days -> update the secret.
from google.colab import userdata
COOKIE_VALUE = userdata.get('CIVITAI_COOKIE')

COMFY_PORT = 8188
COMFY_ROOT = "/content/ComfyUI"
COMFY_LOG  = "/content/comfyui.log"

import subprocess

# Two asserts, not one: a missing secret and a truncated value are different mistakes and a single
# message could not name which one happened.
assert COOKIE_VALUE, "❌ CIVITAI_COOKIE okunamadı — Colab Secrets'a ekle ve 'Notebook access' aç"
assert len(COOKIE_VALUE) > 200, f"❌ CIVITAI_COOKIE çok kısa ({len(COOKIE_VALUE)} karakter) — değer kırpılmış, civitai.red'den __Secure-civ-token'ı çift tıklayıp tamamını kopyala"
print(f"✓ Cookie: Secrets'tan okundu ({len(COOKIE_VALUE)} char)")

print("\n=== GPU ===")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "(nvidia-smi cevap vermedi — Runtime → Change runtime type → A100)")
print("\n=== Disk (~40,1 GB inecek) ===")
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout.strip())

## 2) Ortak Yardımcılar

Tek yerde tanımlanır, sonraki bütün hücreler bunları kullanır (DRY):

- `log(msg, level)` — konsola durum satırı
- `run(cmd, label)` — **tek komut-hatası kapısı**: exit ≠ 0 (HTTP hatası, yarım transfer, dolu disk) veya timeout → `RuntimeError` + komutun **gerçek stderr** kuyruğu
- `check_safetensors(path)` — **ok** (tam) / **partial** (geçerli yarım → resume) / **invalid** (çöp). Beklenen boyutu **dosyanın kendi header'ından** hesaplar (tensör `data_offsets`); sunucuya sormaz
- `head_text(path)` — dosyanın ilk byte'ları, **ham**. Hatalı indirmede sunucunun response gövdesi bu dosyadadır (`curl --fail-with-body`)

Bu dördü kanonik defterden *(`loop_maker/comfy_ui.ipynb`)* birebir geliyor. 4. bölüm bunlara bir tanesini daha ekliyor — **`strip_unreferenced_tail`** — ve sebebi orada yazılı.

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
import os, json, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌", "SKIP": "⏭️ "}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr.

    The single gate for download failures: curl exits non-zero on an HTTP error
    (--fail-with-body), on a transfer that ends before the announced length, and on a full disk.
    """
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request — asking the server is what broke
    here (HF's Xet CDN answers HEAD with 403, whose 48-byte error body was read as the size).

    ok      -> header parses and the file is exactly as long as its header says
    partial -> valid prefix, shorter than expected: safe to resume
    invalid -> empty / error page / longer than its tensors cover: the strict reader refuses it
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors)")

## 3) ComfyUI + Manager + Custom Node'lar (6)

Liste grafiğin kendi **"📋 Features & Requirements"** notundan; altısı da orada adresiyle yazılı.

Manager o listede yok, ayrıca kuruluyor: `MiniMaxH3Director`, `MiniMaxH3Cache`, `MiniMaxChunkFeedForward` gibi node'ların hangi paketten geldiği **bilinmiyor** — grafiğin notu *"routes to the native H3 backend automatically"* diyor, yani güncel ComfyUI'nin kendisinden bekleniyor. Eksik çıkarsa UI'da **Manager → Install Missing Custom Nodes → Restart**.

Klon ya da `pip install` başarısız olursa hücre `RuntimeError` ile durur (fail-loud) — eksik node ileride "node not found" olarak karşımıza çıkmaz.

In [ ]:
%cd /content

# === System deps + ComfyUI ===
# ffmpeg is in the graph's own Requirements note: DaSiWa EnhancedVideoCombine encodes with it.
!apt-get install -y ffmpeg > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt

# === Custom nodes (fail-loud: clone or pip failure -> RuntimeError) ===
import os
os.chdir("/content/ComfyUI/custom_nodes")

# (folder, repo) — trailing comment = what the node provides to this graph
CUSTOM_NODES = [
    ("ComfyUI-Manager",              "https://github.com/ltdrdata/ComfyUI-Manager.git"),                  # install missing nodes from the UI
    ("rgthree-comfy",                "https://github.com/rgthree/rgthree-comfy.git"),                     # Label, Reroute
    ("ComfyUI-KJNodes",              "https://github.com/kijai/ComfyUI-KJNodes.git"),                     # ModelPreviewOverrideKJ -- the taeh3 preview
    ("ComfyUI-GGUF",                 "https://github.com/city96/ComfyUI-GGUF.git"),                       # UnetLoaderGGUF
    ("ComfyUI-DaSiWa-Nodes",         "https://github.com/darksidewalker/ComfyUI-DaSiWa-Nodes.git"),       # Director, SeedControl, Watermark, EnhancedVideoCombine, NodeStatusSwitch
    ("Comfyui-MMH3-UltimateUpscale", "https://github.com/bbaudio-2025/Comfyui-MMH3-UltimateUpscale.git"), # MMH3UltimateUpscale + its split params
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    run(["git", "clone", "--depth", "1", url, name], f"clone {name}", timeout=300)
    if not os.listdir(name):                 # clone reported success but folder is empty
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=600)

os.chdir(COMFY_ROOT)
log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

## 4) Modeller — ~40,1 GB

Sıra: **gated probe** → `.part` durumunu kontrol et → indir (tek bağlantılı `curl`, kaldığı yerden devam) → **damgayı kes** → **doğrula** → gerçek isme rename.

| Klasör | Dosya | Boyut |
|---|---|---|
| `diffusion_models/MiniMaxH3/` | DaSiWa Hybrid Turbo v2, int8 | 21 GB |
| `text_encoders/` | `qwen3vl_32b_minimax_h3_int4_convrot` | 15 GB |
| `vae/MiniMaxH3/` | `minimax_h3_video_vae_int8_convrot` | 3,2 GB |
| `vae/MiniMaxH3/` | `minimax_h3_audio_vae_fp32` | 605 MB |
| `vae_approx/` | `taeh3` | 9,8 MB |
| `loras/` | `MysticXXX_MMH3-V4` · `H3_Motion_BoosterV2` · `H3_VBVR_Pro_attn_only` | 148 + 148 + 63 MB |

**`MiniMaxH3/` bir alt klasör**, ad öneki değil: grafik loader'ına `MiniMaxH3/<dosya>` diyor, ComfyUI da `models/<tür>/MiniMaxH3/` altına bakıyor. Metin kodlayıcı ile `taeh3` kendi klasörlerinin kökünde.

**LoRA'ları grafik adlandırmıyor:** UI'da LoRA yığınından elle seçilirler, o yüzden Civitai'nin kendi dosya adıyla inerler. Hangi sürüm ve neden: `indirilecekler.md` § 7.

### Dönüştürme aracının damgası

Bu MiniMax H3 nicelemelerini üreten araç, çıktısının **sonuna bir imza yazıyor**: son tensörden sonra bir satır ASCII, `L2P_bypass_<kaynak dosya>_<unix zaman>` biçiminde. Sonuç, aynı dosyanın **hangi okuyucuya denk geldiğine göre** açılması ya da açılmaması:

| Okuyucu | Son tensörden sonrası | Sonuç |
|---|---|---|
| ComfyUI'nin kendi `load_safetensors` *(DynamicVRAM açıkken)* | `data_offsets`'i yürür, görmez | açar |
| Rust `safetensors` *(normalde)* | dosyanın tamamının kapsanmasını şart koşar | **reddeder** |

Üstelik reddin cümlesi, **gerçekten kesik** bir dosyanınkiyle aynı: *"incomplete metadata, file not fully covered"*. Kaynak: [Comfy-Org/ComfyUI issue #15602](https://github.com/Comfy-Org/ComfyUI/issues/15602).

`strip_unreferenced_tail` her kontrolden **önce** bu kuyruğu kesiyor ve **ne kestiğini basıyor**. Kesmek güvenli, çünkü o baytlar hiçbir tensöre ait değil: header'ın saydığı her tensör onlardan önce bitiyor, yani oraya kadar gelmiş bir dosya bütün verisini taşıyor. Kesilmiş dosya her iki okuyucuyla da açılır — DynamicVRAM ayarına bağlı kalmayız.

### Gerisi

**Doğrulama ağa soru sormaz.** Safetensors = `[8 byte header uzunluğu][header JSON][tensör verisi]`; header'daki `data_offsets` verinin bittiği yeri söyler → beklenen boyut **dosyanın kendisinden** çıkar.

**Gated erişim ağır indirmeden önce doğrulanır** (ilk 1 KB): cookie ölmüşse 21 GB'lık checkpoint'e başlamadan, Civitai'nin **gerçek yanıtıyla** durur.

**Hata olursa durulur, ham çıktı basılır, hiçbir şey silinmez.** `curl --fail-with-body` sayesinde HTTP hatasında response gövdesi `.part`'a yazılır → hata mesajında curl'ün stderr'i + sunucunun **kendi yanıtı** görünür.

> HF büyük dosyaları **Xet** depoda: imzalı CDN URL'i paralel byte-range isteklerine **403** verir, HEAD'i de 403 ile reddeder (GET çalışır) → tek bağlantılı `curl` GET (`parallel=False`). Civitai tarafı da curl: `aria2c` cookie'yi yönlendirmede B2 deposuna taşıyıp 403 alıyor, curl cross-host'ta düşürüp geçiyor.

In [ ]:
import os, glob

# === Target folders ===
# MiniMaxH3/ is a SUBFOLDER, not part of the name: the graph asks its loaders for
# "MiniMaxH3/<file>", so ComfyUI looks under models/<kind>/MiniMaxH3/. The text encoder and the
# preview TAE carry no prefix and sit at the root of their own folder.
DIFF = f"{COMFY_ROOT}/models/diffusion_models/MiniMaxH3"
TENC = f"{COMFY_ROOT}/models/text_encoders"
VAE  = f"{COMFY_ROOT}/models/vae/MiniMaxH3"
TAE  = f"{COMFY_ROOT}/models/vae_approx"
LORA = f"{COMFY_ROOT}/models/loras"
for d in (DIFF, TENC, VAE, TAE, LORA):
    os.makedirs(d, exist_ok=True)

def strip_unreferenced_tail(path):
    """Cut bytes that no tensor claims off the end of a safetensors file; return what was cut.

    The tool behind these MiniMax H3 quants stamps its output: a line of ASCII after the last
    tensor, shaped like L2P_bypass_<source file>_<unix time>. ComfyUI's own parser walks
    data_offsets and never sees it, but the Rust safetensors parser demands that the tensors cover
    the whole file and refuses to open it -- so the same checkpoint loads or does not depending on
    which reader runs (Comfy-Org/ComfyUI issue 15602). Its refusal also reads exactly like a
    genuinely truncated file, which is what made this expensive to pin down.

    Cutting is safe precisely because the bytes are unreferenced: every tensor the header declares
    ends before them, so a file long enough to reach here is holding all of its data, and what
    comes off is what no loader would have read. It is returned rather than dropped so the caller
    can print what it threw away.

    b"" when the file is short, unreadable or already exactly covered. This repairs one thing and
    judges nothing -- the verdict stays with check_safetensors.
    """
    try:
        size = os.path.getsize(path)
        with open(path, "rb") as f:
            header_len = struct.unpack("<Q", f.read(8))[0]
            if not (0 < header_len < 200_000_000) or 8 + header_len > size:
                return b""
            header = json.loads(f.read(header_len).decode("utf-8"))
    except (OSError, ValueError, UnicodeDecodeError, struct.error):
        return b""

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:
        return b""

    expected = 8 + header_len + max(ends)
    if size <= expected:
        return b""

    with open(path, "rb") as f:
        f.seek(expected)
        tail = f.read()
    os.truncate(path, expected)
    return tail

def fetch(url, target_dir, filename, label, *, parallel=False, headers=None):
    """Download + validate a model; anything invalid stops the run (fail-loud, nothing deleted).

    parallel=True  -> aria2c (many connections; fine for plain S3/CDN hosts)
    parallel=False -> single-connection curl (HF Xet rejects parallel byte ranges with 403)

    Downloads land in <target>.part and are renamed only once check_safetensors says "ok", so
    ComfyUI never sees a half-written file under the real model name. A complete .part is never
    handed to curl, so resume cannot hit "range past EOF" (416).

    A stamped tail is cut before every check, never after: a .part that is complete but stamped
    then reads as "ok", the download is skipped, and curl is never asked to resume onto a file it
    would stamp again.

    On failure the raw HTTP exchange is printed, not a summary of it: curl runs with
    --fail-with-body (non-zero exit, but the response body is kept instead of discarded) and -D
    (every response header of the redirect chain), so the error carries curl's stderr + the
    server's headers + the server's body verbatim.
    """
    target = os.path.join(target_dir, filename)
    part = target + ".part"
    hdrs = f"/tmp/{filename}.headers"

    def strip_and_report(p):
        """Repair, loudly. A silent truncation is the one thing worse than the error it fixes."""
        cut = strip_unreferenced_tail(p)
        if cut:
            log(f"{label}: kuyrukta {len(cut)} sahipsiz bayt vardı, atıldı → {cut[:200]!r}", "WARN")

    if os.path.exists(target):
        strip_and_report(target)        # a file renamed by an older run can still carry its stamp
        state, msg = check_safetensors(target)
        if state == "ok":
            log(f"{label}: zaten var ({msg})")
            return
        raise RuntimeError(f"{label}: {state} — {msg}\n{target}\n--- file head ---\n{head_text(target)}")

    resume = False
    if os.path.exists(part):
        strip_and_report(part)
        state, msg = check_safetensors(part)
        if state == "invalid":
            # Resuming onto garbage would append good bytes to it and hide the problem.
            raise RuntimeError(f"{label}: .part {state} — {msg}\n{part}\n--- file head ---\n{head_text(part)}")
        if state == "ok":
            log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
        else:
            log(f"{label}: .part'tan devam ({msg})")
            resume = True

    if not os.path.exists(part) or resume:
        log(f"{label}: iniyor")
        if parallel:
            cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--continue=true",
                   "--console-log-level=warn", "--auto-file-renaming=false", "--allow-overwrite=true",
                   "-d", target_dir, "-o", os.path.basename(part)]
            if headers:
                cmd += ["--header", headers]
        else:
            cmd = ["curl", "-L", "-C", "-", "--fail-with-body", "--retry", "5", "--retry-delay", "5",
                   "--retry-all-errors", "-D", hdrs, "-o", part]
            if headers:
                cmd += ["-H", headers]
        cmd.append(url)
        try:
            run(cmd, label, timeout=7200)   # HTTP error / short transfer / full disk -> here
        except RuntimeError as e:
            raise RuntimeError(
                f"{e}\n{url}\n"
                f"--- response headers ---\n{head_text(hdrs)}\n"
                f"--- response body ---\n{head_text(part)}"
            ) from None

    strip_and_report(part)
    state, msg = check_safetensors(part)
    if state != "ok":
        raise RuntimeError(f"{label}: {state} — {msg}\n{part}\n{url}\n"
                           f"--- response headers ---\n{head_text(hdrs)}\n"
                           f"--- file head ---\n{head_text(part)}")
    os.replace(part, target)
    log(f"{label}: indirildi ve doğrulandı ({msg})", "OK")

# Civitai auth: session cookie ONLY. A ?token= API key authenticates the request as that key's
# account -> creator-gated assets answer 401.
# Host = civitai.RED: the cookie is same-origin there. Sending it to .com is cross-domain and
# returns the login+turnstile page instead of the file.
def civitai_url(version_id):
    return f"https://civitai.red/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civ-token={COOKIE_VALUE}"

def civitai_probe(version_id, label):
    """Fail-fast: range-download the first 1KB to verify gated access BEFORE a 21 GiB checkpoint.
    On non-2xx or a login wall, surface Civitai's ACTUAL response body -- no hardcoded guesses.
    """
    out = "/content/_probe.bin"
    code = (run(["curl", "-sL", "--max-time", "60", "-r", "0-1023",
                 "-H", cookie_header(), "-w", "%{http_code}", "-o", out,
                 civitai_url(version_id)], f"probe {label}") or "").strip()[-3:]
    body = b""
    if os.path.exists(out):
        with open(out, "rb") as f:
            body = f.read(512)
        os.remove(out)
    # success = 2xx AND the body is real binary (safetensors), not an HTML/JSON error page
    if code.startswith("2") and not body.startswith(b"<") and not body.startswith(b'{"'):
        log(f"{label}: erişim OK", "OK")
        return
    raise RuntimeError(f"❌ {label}: HTTP {code} — Civitai yanıtı: "
                       f"{body.decode('utf-8', 'replace').strip() or '(boş gövde — binary değil)'}")

# === HuggingFace models ===
# Every address below is the one the graph's own "Model Links" note gives; which file and why:
# indirilecekler.md. Nothing is derived here a second time.
HF = "https://huggingface.co"

HF_MODELS = [
    # (url, target_dir, filename, label)
    (f"{HF}/Abiray/MiniMax-H3-GGUF/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int4_convrot.safetensors",
     TENC, "qwen3vl_32b_minimax_h3_int4_convrot.safetensors", "CLIP Qwen3-VL 32B int4 (15 GB)"),
    (f"{HF}/Kijai/MiniMax-H3-experimental/resolve/main/minimax_h3_video_vae_int8_convrot.safetensors",
     VAE,  "minimax_h3_video_vae_int8_convrot.safetensors",   "Video VAE int8 (3,2 GB)"),
    (f"{HF}/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",
     VAE,  "minimax_h3_audio_vae_fp32.safetensors",           "Ses VAE fp32 (605 MB)"),
    (f"{HF}/Kijai/MiniMax-H3-TAE/resolve/main/vae_approx/taeh3.safetensors",
     TAE,  "taeh3.safetensors",                               "TAE önizleme (9,8 MB)"),
]

# === Civitai gated model ===
# Version 3314686 is "DaSiWa Hybrid Turbo v2" -- the turbo one, which is what the graph ships
# configured for (euler, 8 steps). Civitai serves it under its own name
# (DasiwaMinimaxH3_dasiwaHybridTurboV2_*.safetensors); both UNET slots of the graph ask for the
# name below, so it lands under that one. Why this version and not a neighbour: indirilecekler.md § 6.
DASIWA_VERSION = 3314686
DASIWA_FILE = ("dasiwa_minimax_h3_ref2va_v2_pruned_hybrid_turbo_int8_"
               "row-wise_convrot_runtime_mixed.safetensors")

# === Civitai gated LoRAs ===
# The graph names none of them: they are picked by hand in its LoRA stack (DaSiWa_LTX2LoraLoader),
# so they land under Civitai's own primary file names. The addresses are the user's, and the
# versions the newest ones on each page -- indirilecekler.md § 7. No speed LoRA here: Turbo v2
# already carries its distillation, and whether to add one is an open question in the backlog.
CIVITAI_LORAS = [
    # (version_id, filename, label)
    (3266628, "MysticXXX_MMH3-V4.safetensors",     "Mystic XXX v4.0 (148 MB)"),
    (3228867, "H3_Motion_BoosterV2.safetensors",   "Motion Booster V0.2 (148 MB)"),
    (3306139, "H3_VBVR_Pro_attn_only.safetensors", "VBVR Pro (63 MB)"),
]

# 1) Fail-fast: verify gated access before spending the download time
civitai_probe(DASIWA_VERSION, "DaSiWa Hybrid Turbo v2")
for vid, fn, label in CIVITAI_LORAS:
    civitai_probe(vid, label)

# 2) HuggingFace -- single connection (Xet)
for url, d, fn, label in HF_MODELS:
    fetch(url, d, fn, label, parallel=False)

# 3) Civitai -- curl as well: aria2c forwards the cookie to the B2 store on redirect and gets 403,
#    curl drops it cross-host and gets through.
fetch(civitai_url(DASIWA_VERSION), DIFF, DASIWA_FILE,
      "DaSiWa Hybrid Turbo v2 int8 (21 GB)", parallel=False, headers=cookie_header())
for vid, fn, label in CIVITAI_LORAS:
    fetch(civitai_url(vid), LORA, fn, label, parallel=False, headers=cookie_header())

# === Summary (reaching here means every model downloaded + validated) ===
print("")
for d, title in ((DIFF, "diffusion_models/MiniMaxH3"), (TENC, "text_encoders"),
                 (VAE, "vae/MiniMaxH3"), (TAE, "vae_approx"), (LORA, "loras")):
    print(f"📂 {title}/")
    for f in sorted(glob.glob(f"{d}/*.safetensors")):
        print(f"   {human(os.path.getsize(f)):>8s}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## 5) Başlat + cloudflared Tünel

ComfyUI arka planda başlar (90 sn içinde `/system_stats` cevap vermezse log'un son 30 satırını basıp fail-loud), tünel linki basılır, sonra hücre **bilerek açık kalır** — biterse Colab runtime'ı idle sayıp tüneli öldürür.

In [ ]:
import subprocess, time, urllib.request, re, os

if not os.path.isfile("/content/cloudflared"):
    run(["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], "cloudflared")
    run(["chmod", "+x", "/content/cloudflared"], "chmod cloudflared")

# Re-run safety: kill the previous ComfyUI + tunnel before starting new ones
subprocess.run(["pkill", "-f", "main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

# --enable-manager: on current ComfyUI the Manager is OFF without this flag. The workflow is loaded
# by hand and some of its H3 nodes are unaccounted for, so the Manager has to stay reachable.
logf = open(COMFY_LOG, "w")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT), "--enable-manager"],
                 cwd=COMFY_ROOT, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(COMFY_LOG).readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")
log(f"ComfyUI ayakta ({(i + 1) * 2}s)", "OK")

# cloudflared output goes to a file, not a pipe: an unread pipe fills up and blocks the process.
tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read()) if os.path.exists(tunlog) else None
    if m:
        link = m.group(0)
        break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 ComfyUI linki: {link}\n")
print("⬆️  Linke gir → bu klasördeki workflow.json'u kendi bilgisayarından sürükle-bırak")
print("    Grafik I2VA modunda açılır: Director'e bir fotoğraf ver, prompt'u yaz, Queue Prompt (Ctrl+Enter)")
print("\n    Settings panelinde seçili olmalı:")
print("      UNET (iki yuva da)  MiniMaxH3/dasiwa_..._hybrid_turbo_int8_...")
print("      CLIP                qwen3vl_32b_minimax_h3_int4_convrot")
print("      Video VAE           MiniMaxH3/minimax_h3_video_vae_int8_convrot")
print("      Ses VAE             MiniMaxH3/minimax_h3_audio_vae_fp32")
print("      Sampler euler, 8 adım, shift video 6 / audio 3 — turbo checkpoint bunu ister")
print("\n⚠️  'MiniMax H3 Cache'i ilk denemede açma — grafiğin kendi uyarısı: ghost/morph yapabilir")
print("⚠️  VRAM yetmezse: Director → '⚙️ Chunking' (MiniMax H3 Chunk FeedForward)")
print("    Eksik node olursa: Manager → Install Missing Custom Nodes → Restart")
print("    Beğenirsen: Workflow → Export (API) — madde 213 queen-editor'e onu koyacak\n")

# === Keep the cell OPEN (critical) ===
# ComfyUI + the tunnel run in the background. If this cell ENDS, Colab can call the runtime idle
# and disconnect -> ComfyUI + link die. Streaming the log keeps the cell in the foreground and
# shows generation progress when Queue Prompt is pressed in the UI.
# To stop: interrupt this cell (■) or Runtime -> Disconnect.
print("📡 ComfyUI çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", COMFY_LOG])
except KeyboardInterrupt:
    log("Hücre durduruldu — ComfyUI hâlâ arka planda çalışıyor (yeni link için bu hücreyi tekrar çalıştır).", "WARN")